In [ ]:
%pip install eyepop==3.12.0
%pip install google-genai

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
GEMINI_API_KEY=getpass.getpass('Enter your API KEY: ')

In [6]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Abilities

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.detect-vehicle-damage",
        description="Identify if all workers are wearing their helmets",
        worker_release="qwen3-instruct",
        text_prompt="""
          Determine the primary content of the input and assign exactly one label from the following list: ['Damage', 'No_Damage'].

          CRITICAL: You must explicitly rule out natural lines of a car and shadows/lighting before classifying a frame as 'Damage'.

          Choose 'No_Damage' if the vehicle is intact, OR if the perceived anomaly could reasonably be explained by:

          Lighting: Harsh shadows, glare, or natural light reflections on the curves of the car.

          Geometry: Intended styling body-lines, natural creasing, or standard manufacturing panel gaps (seams outlining doors, the hood, or the trunk).

          Minor Wear: Dirt, water spots, or superficial clear-coat scuffs that do not require auto body repair.

          Choose 'Damage' ONLY if you can definitively confirm clear, actionable physical defects that cannot be explained by lighting or car design. This requires unambiguous evidence of:

          Dents or crushed/crumpled body panels.

          Shattered or cracked glass (windows, mirrors, lights).

          Deep scratches that clearly penetrate the paint.

          Broken, severely misaligned, or detached parts (e.g., a hanging bumper).

          Return only the single label.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=3,
            image_size=512
        ),
        is_public=False
    ),
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.vehicle-damage",
        description="Describe the vehicle damage and provide an estimated quote of how much it would cost to fix",
        worker_release="qwen3-instruct",
        text_prompt="""
        You are an expert auto body estimator and insurance adjuster. Analyze this visual evidence of vehicle damage and extract the following information.

        Vehicle Identification: Identify the make, model, and vehicle class based on its visual design. If unsure, state GUESS: and provide your best visual guess.

        Damage Location: Specify the exact location of the damage on the vehicle (e.g., rear passenger-side bumper, driver-side front door).

        Damage Description: Provide a detailed description of the severity and type of damage (e.g., severe structural crumpling, shattered taillight assembly, deep paint scratches).

        Estimated Repair Cost: Based on the identified vehicle type and the severity of the damage, provide an estimated repair cost range in US Dollars. Factor in the likelihood of parts replacement versus simple dent repair.

        Return your response only as a JSON object with the keys: vehicle_identification, damage_location, damage_description, and estimated_cost.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=200,
            fps=3,
            image_size=512
        ),
        is_public=False
    )
]

### Create Abilities

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from google import genai
from google.genai import types
from pathlib import Path

pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.find-events.detect-vehicle-damage:latest",
        forward=FullForward(
            targets=[ForwardComponent(
                forward=FullForward(
                    includeClasses=["Damage"],
                    targets=[InferenceComponent(
                        ability=f"{NAMESPACE_PREFIX}.describe.vehicle-damage:latest",
                        categoryName="damage_description"
                    )]
                )
            )]
        )
    )
])
with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)
    sample_vid_path = Path("/content/sample_vid.mp4")
    job = endpoint.upload(sample_vid_path)
    accumulated_reports = []
    while result := job.predict():
      classes = result.get("classes", [])
      has_damage = any(c.get("classLabel") == "Damage" for c in classes)

      if has_damage:
        texts = result.get("texts", [])
        for text_item in texts:
            # We grab the raw JSON string from the frame
            raw_text = text_item.get("text", "")
            accumulated_reports.append(raw_text)

all_reports_string = "\n---\n".join(accumulated_reports)

prompt = f"""
You are an expert insurance adjuster and auto body estimator.
I am providing you with multiple frame-by-frame AI analyses from a video walk-around of a single damaged vehicle.

Because these are individual frame analyses from a video, the vision model may hallucinate, misidentify parts, or contradict itself across frames. Your primary job is to filter out the noise and determine the true damage by finding the consensus.

Please read through all the individual frame reports and synthesize them into ONE final, cohesive damage report based on these strict rules:

Count the "Votes": Treat each frame as a vote for what vehicle it is and what is damaged.

Report Only the Consensus: Cross-reference the damage locations and descriptions. ONLY include the specific damaged parts and vehicle details that are reported consistently in the majority of the frames.

Discard Outliers: Completely ignore damages, parts, or vehicle models that only appear in a minority of frames. Do not merge every single damage mentioned into one massive list. If 8 frames say the damage is to the passenger door, and 2 frames mention a bumper, exclude the bumper entirely.

Provide the final report in this exact structure:
* Make and Model: [Synthesize the most likely vehicle]
* Overall Severity: [Brief 1-2 sentence summary of the incident]
* Specific Damages by Part: [Bullet points listing each affected car part and what is wrong with it]
* Estimated Cost Range: [A logical dollar range based on the provided frame estimates]

Here is the raw frame-by-frame data:
{all_reports_string}
"""

# Create the model client
client = genai.Client(api_key=GEMINI_API_KEY)

# Generate content
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

# 7. Print the final synthesized report
print("\n=== FINAL DAMAGE REPORT ===")

print(response.text)